# Libraries

In [1]:
import pandas as pd
from pathlib import Path

# Try to use rapidfuzz if available; otherwise fallback to difflib
try:
    from rapidfuzz import process, fuzz
    USE_RAPIDFUZZ = True
except Exception:
    import difflib
    from difflib import SequenceMatcher
    USE_RAPIDFUZZ = False


# Combining 2023 and 2024 PRC passing rates

In [105]:
# Loading both files
file_2024 = "PRC 2024.csv"
file_2023 = "PRC 2023.csv"

df_2024 = pd.read_csv(file_2024)
df_2023 = pd.read_csv(file_2023)

# Basic info about each file
df_2024_info = df_2024.head(), df_2024.shape, df_2024.columns.tolist()
df_2023_info = df_2023.head(), df_2023.shape, df_2023.columns.tolist()

df_2024_info, df_2023_info

((             job_sector  year  passers  examinees  passing_rate  \
  0           Accountancy  2024     3058      10136         30.17   
  1           Agriculture  2024     3628       7144         50.78   
  2          Architecture  2024     2094       3370         62.14   
  3  Chemical Engineering  2024      688        951         72.34   
  4             Chemistry  2024      592       1088         54.41   
  
                                                source  
  0  https://www.thesummitexpress.com/2024/12/cpale...  
  1  https://www.thesummitexpress.com/2024/11/full-...  
  2  https://mb.com.ph/2024/6/20/pup-tops-june-2024...  
  3  https://www.philstar.com/banat/imong-kapalaran...  
  4  https://mb.com.ph/2024/10/24/october-2024-chem...  ,
  (46, 6),
  ['job_sector', 'year', 'passers', 'examinees', 'passing_rate', 'source']),
 (             job_sector  year  passers  examinees  passing_rate  \
  0           Accountancy  2023     2740       8734         31.37   
  1           

Removing the 7th column of `PRC 2023.csv` because it's junk text from the source links.

In [106]:
df_2023_clean = df_2023.drop(columns=["Unnamed: 6"])

Merging the yearly results side-by-side based on the job sector

In [107]:
merged_df = pd.merge(df_2023_clean, df_2024, on="job_sector", suffixes=("_2023", "_2024"))

# Removing source columns
merged_df = merged_df.drop(columns=["source_2023", "source_2024"])

# Renaming job_sector to Job Sector in preparation for combining with MCA JOBLIST.csv
merged_df = merged_df.rename(columns={'job_sector': 'Job Sector'})

CSV file for `merged_df`

In [108]:
merged_df.to_csv("2023 AND 2024 COMBINED.csv", index=False)


# Handling the Mapping Between the Two Datasets

If you look into `MCA JOBLIST.csv`, you will notice that the formatting of the job sectors is different to that of `2023 AND 2024 COMBINED.csv`:

- PRC data has sectors like “Agriculture,” “Forestry,” “Fisheries Technology” as separate rows.
- On the other hand, MCA JOBLIST data has a broader category like “Agriculture, Forestry, and Fishing” that combines all three.

With that said, the way I handled it was by creating a mapping table that links MCA job sectors to the PRC sectors. Going back to the “Agriculture, Forestry, and Fishing” entry in MCA Joblist, it would find all the “Agriculture,” “Forestry,” “Fisheries Technology” entries within the PRC file. 

Loading the files + functions

In [ ]:
import pandas as pd
from pathlib import Path

# ---- Config
THRESHOLD = None  
OUT_ENRICHED = Path("MCA Job List Tagged FINAL-checkpoint.csv")
OUT_MATCHES = Path("sector_matches.csv")

# ---- Helpers
def clean_str(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().lower().replace("&", "and").split())

def recompute_rate(passers, examinees):
    try:
        p = float(passers)
        e = float(examinees)
    except Exception:
        return float("nan")
    if e is None or e == 0:
        return float("nan")
    return round(p / e * 100, 2)  # percentages, to 2 decimals

# ---- Load files
combined_path = "2023 AND 2024 COMBINED.csv"
joblist_path = "MCA JOBLIST.csv"

df_comb = pd.read_csv(combined_path)
df_jl = pd.read_csv(joblist_path)

In [110]:
# ---- Standardize combined columns
# Expect "Job Sector" + the six metric columns; if not present, try to infer
lower_map = {c.lower().strip(): c for c in df_comb.columns}

def find_col(*cands):
    for c in cands:
        if c.lower() in lower_map:
            return lower_map[c.lower()]
    return None

sector_col_c = find_col("Job Sector", "job_sector", "sector")

pass23 = find_col("passers_2023")
exam23 = find_col("examinees_2023")
rate23 = find_col("passing_rate_2023")
pass24 = find_col("passers_2024")
exam24 = find_col("examinees_2024")
rate24 = find_col("passing_rate_2024")

need_cols = [sector_col_c, pass23, exam23, rate23, pass24, exam24, rate24]
need_cols = [c for c in need_cols if c is not None]

dfc = df_comb[need_cols].copy()
rename_map = {}
if sector_col_c: rename_map[sector_col_c] = "Job Sector"
if pass23: rename_map[pass23] = "passers_2023"
if exam23: rename_map[exam23] = "examinees_2023"
if rate23: rename_map[rate23] = "passing_rate_2023"
if pass24: rename_map[pass24] = "passers_2024"
if exam24: rename_map[exam24] = "examinees_2024"
if rate24: rename_map[rate24] = "passing_rate_2024"
dfc = dfc.rename(columns=rename_map)

# Ensure numeric
for c in ["passers_2023","examinees_2023","passing_rate_2023","passers_2024","examinees_2024","passing_rate_2024"]:
    if c in dfc.columns:
        dfc[c] = pd.to_numeric(dfc[c], errors="coerce")

In [111]:
# If passing rates missing, compute later after aggregation
# ---- Aggregate by Job Sector (in case of duplicates) and recompute rates
dfc["_key"] = dfc["Job Sector"].map(clean_str)
agg = dfc.groupby("_key", as_index=False).agg({
    "Job Sector": "first",
    "passers_2023": "sum",
    "examinees_2023": "sum",
    "passers_2024": "sum",
    "examinees_2024": "sum",
})

# Recompute rates as percentages
agg["passing_rate_2023"] = [recompute_rate(p, e) for p, e in zip(agg["passers_2023"], agg["examinees_2023"])]
agg["passing_rate_2024"] = [recompute_rate(p, e) for p, e in zip(agg["passers_2024"], agg["examinees_2024"])]

# ---- Prepare joblist
# Find PRC column
prc_col = None
for c in df_jl.columns:
    cl = c.lower().strip()
    if cl.startswith("hei with prc"):  # matches "HEI with PRC (Professional Regulation Commission) Exam"
        prc_col = c
        break
if prc_col is None:
    raise ValueError("Cannot find the PRC flag column in MCA JOBLIST.csv. Expected a column starting with 'HEI with PRC'.")

# Ensure Job Sector exists
if "Job Sector" not in df_jl.columns:
    # Try to find close variant
    for c in df_jl.columns:
        if "job sector" in c.lower():
            df_jl = df_jl.rename(columns={c: "Job Sector"})
            break
if "Job Sector" not in df_jl.columns:
    raise ValueError("Cannot find 'Job Sector' column in MCA JOBLIST.csv.")

In [112]:
# ---- Step 1: Add the 6 columns if missing
for col in ["passers_2023","examinees_2023","passing_rate_2023","passers_2024","examinees_2024","passing_rate_2024"]:
    if col not in df_jl.columns:
        df_jl[col] = pd.NA

# ---- Step 2: Fuzzy replace Job Sector ONLY if PRC == Yes
try:
    from rapidfuzz import process, fuzz
    use_rf = True
except Exception:
    use_rf = False
    import difflib

combined_sector_list = (
    agg["Job Sector"]
    .dropna()
    .astype(str)
    .str.strip()
    .drop_duplicates()
    .tolist()
)

# Fuzzy match function
def best_match(query, choices):
    q = str(query)
    if use_rf:
        match = process.extractOne(q, choices, scorer=fuzz.token_set_ratio, processor=clean_str)
        if match is None:
            return q, 0.0
        choice, score, _ = match
        return choice, float(score)
    else:
        # difflib fallback
        cleaned_choices = [clean_str(c) for c in choices]
        q_clean = clean_str(q)
        scores = [difflib.SequenceMatcher(None, q_clean, c).ratio()*100.0 for c in cleaned_choices]
        if not scores:
            return q, 0.0
        idx = max(range(len(scores)), key=lambda i: scores[i])
        return choices[idx], float(scores[idx])

# Apply conditional overwrite
matches = []
prc_yes = df_jl[prc_col].astype(str).str.strip().str.lower().eq("yes")

for i, row in df_jl.iterrows():
    cur = row["Job Sector"]
    if prc_yes.iloc[i]:
        best, score = best_match(cur, combined_sector_list)
        if THRESHOLD is None or score >= THRESHOLD:
            df_jl.at[i, "Job Sector"] = best
        matches.append({
            "row_index": i,
            "original_job_sector": cur,
            "new_job_sector": df_jl.at[i, "Job Sector"],
            "score": round(score, 1)
        })
    else:
        matches.append({
            "row_index": i,
            "original_job_sector": cur,
            "new_job_sector": cur,
            "score": None
        })

matches_df = pd.DataFrame(matches)
matches_df.to_csv(OUT_MATCHES, index=False)

# ---- Step 3: Transfer data for matching Job Sector
df_jl["_key"] = df_jl["Job Sector"].map(clean_str)

enriched = df_jl.merge(agg.drop(columns=["Job Sector"]), on="_key", how="left", suffixes=("", "_agg"))

# Copy over aggregated values into the six columns
for col in ["passers_2023","examinees_2023","passing_rate_2023","passers_2024","examinees_2024","passing_rate_2024"]:
    if col in enriched.columns and f"{col}" in agg.columns:
        # If the merge created a column with the same name (no suffix), it's already correct.
        # But ensure we take from the aggregated columns where present.
        pass

# When merging, our aggregated columns exist exactly once; assign them into df_jl's six target columns
for col in ["passers_2023","examinees_2023","passing_rate_2023","passers_2024","examinees_2024","passing_rate_2024"]:
    if col in enriched.columns:
        # Already present from merge; nothing to do
        continue
# (No-op because our merge kept the same names; if columns didn't exist they were created earlier and will now be present.)

# Drop helper key
if "_key" in enriched.columns:
    enriched = enriched.drop(columns=["_key"])

print("Outputs created:")
print(OUT_MATCHES)

Outputs created:
sector_matches.csv


In [113]:
# Save enriched file
# enriched.to_csv(OUT_ENRICHED, index=False)

Then I manually changed some of the entries if they didn't make sense.

# MANUAL CHANGES

## Functions

In [2]:
# Interactive, row-specific updater for MCA JOBLIST_enriched.csv
# - You specify (row_index, new_job_sector)
# - It updates the row's 'Job Sector' and pulls mapped fields from the source CSV
# - You can stage many edits, preview diffs, then save once at the end
#
# This cell defines helper functions and prepares a working copy. It does NOT save the file.

import pandas as pd
from pathlib import Path

SRC_PATH = Path("2023 AND 2024 COMBINED.csv")
TGT_PATH = Path("MCA Job List Tagged FINAL-checkpoint.csv")

# Load
src = pd.read_csv(SRC_PATH)
tgt_original = pd.read_csv(TGT_PATH)

# Work on a copy so we don't mutate the original until you call save_updated(...)
work = tgt_original.copy()

# Clean column names (defensive)
src.columns = src.columns.str.strip()
work.columns = work.columns.str.strip()

KEY = "Job Sector"

# Mapping: target <- source
MAPPING = {
    "passers_2023_agg": "passers_2023",
    "examinees_2023_agg": "examinees_2023",
    "passers_2024_agg": "passers_2024",
    "examinees_2024_agg": "examinees_2024",
    "passing_rate_2023_agg": "passing_rate_2023",
    "passing_rate_2024_agg": "passing_rate_2024",
}

# Build a quick lookup from source by Job Sector
if KEY not in src.columns:
    raise ValueError(f'"{KEY}" not found in source file. Found: {list(src.columns)}')

src_lookup = src.set_index(KEY)

# Track which row indices have been edited
edited_rows = set()

def _pull_values_for_sector(sector: str) -> dict:
    """Return a dict of target_col -> value from the source for a given sector. Raises if sector not found."""
    if sector not in src_lookup.index:
        raise KeyError(f"Job Sector '{sector}' not found in source file.")
    row = src_lookup.loc[sector]
    values = {}
    for tgt_col, src_col in MAPPING.items():
        if src_col not in src_lookup.columns:
            raise KeyError(f"Source column '{src_col}' missing in source file.")
        values[tgt_col] = row[src_col]
    return values

def set_row(row_index: int, new_job_sector: str):
    """Set the row's Job Sector and copy mapped values from the source for that sector. Stages the change."""
    if row_index < 0 or row_index >= len(work):
        raise IndexError(f"row_index {row_index} out of bounds (0..{len(work)-1}).")
    # Pull values (will raise if sector missing)
    vals = _pull_values_for_sector(new_job_sector)
    # Update the key and mapped columns
    work.at[row_index, KEY] = new_job_sector
    for tgt_col, value in vals.items():
        # Ensure column exists; create if missing
        if tgt_col not in work.columns:
            work[tgt_col] = pd.NA
        work.at[row_index, tgt_col] = value
    edited_rows.add(row_index)

def set_row_excel(excel_row_number: int, new_job_sector: str):
    """
    Update using Excel-style row numbers (the ones shown in Excel on the left).
    Example: if Excel shows row 25, call set_row_excel(25, "Agriculture").
    Internally it converts to the right pandas row_index.
    """
    # Excel row 7 is pandas row_index 0 (because header is row 1–6 in your file)
    row_index = excel_row_number - 2
    set_row(row_index, new_job_sector)

def apply_changes(changes):
    """
    Apply a list of changes. Example formats accepted:
      - [(12, 'Accounting'), (45, 'Nursing')]
      - [{'row_index': 12, 'job_sector': 'Accounting'}, {'row_index':45, 'job_sector':'Nursing'}]
    """
    if isinstance(changes, pd.DataFrame):
        # Expect columns: row_index, job_sector
        for _, r in changes.iterrows():
            set_row(int(r["row_index"]), str(r["job_sector"]))
        return

    for item in changes:
        if isinstance(item, dict):
            set_row(int(item["row_index"]), str(item["job_sector"]))
        else:
            idx, sector = item
            set_row(int(idx), str(sector))

def preview_changes(limit=25):
    """Show a diff-style preview for the rows you've edited."""
    if not edited_rows:
        return pd.DataFrame(columns=["row_index", KEY] + list(MAPPING.keys()))
    rows = sorted(list(edited_rows))
    cols = [KEY] + list(MAPPING.keys())
    before = tgt_original.loc[rows, cols].copy()
    after = work.loc[rows, cols].copy()
    before.columns = [f"{c}__before" for c in before.columns]
    after.columns = [f"{c}__after" for c in after.columns]
    out = pd.concat([before.reset_index().rename(columns={"index":"row_index"}),
                     after.reset_index(drop=True)], axis=1)
    # Display for user
    display_dataframe_to_user("Staged changes (before vs after)", out.head(limit))
    return out

def save_updated(out_path="MCA_JOBLIST_enriched_UPDATED_rows.csv"):
    """Write the staged 'work' DataFrame to CSV. Call this only when you're done editing."""
    path = Path(out_path)
    work.to_csv(path, index=False)
    return str(path)

# Provide a small editable template the user can fill in (row_index, job_sector)
template = pd.DataFrame({"row_index":[0], "job_sector":["<type new sector here>"]})
template_path = Path("pending_changes_template.csv")
template.to_csv(template_path, index=False)

# Show a tiny guide + a peek at current rows so the user can find indices
peek = work[[KEY]].reset_index().rename(columns={"index":"row_index"}).head(25)

print("Functions ready: set_row(row_index, new_job_sector), apply_changes(changes), preview_changes(limit=25), save_updated(path).")
print("Template saved at:", template_path)


Functions ready: set_row(row_index, new_job_sector), apply_changes(changes), preview_changes(limit=25), save_updated(path).
Template saved at: pending_changes_template.csv


## Changes

In [6]:
set_row_excel(25, "Agriculture")

In [7]:
set_row_excel(51, "Fisheries Technology")

In [8]:
set_row_excel(52, "Professional Teachers")

In [9]:
set_row_excel(56, "Architecture")

In [10]:
set_row_excel(57, "Library Science")

In [11]:
set_row_excel(123, "Library Science")

In [12]:
set_row_excel(148, "Library Science")
set_row_excel(150, "Accountancy")
set_row_excel(151, "Accountancy")

In [13]:
set_row_excel(159, "Chemistry")
set_row_excel(160, "Chemical Engineering")
set_row_excel(163, "Chemical Engineering")


In [14]:
set_row_excel(170, "Electronics Engineering")
set_row_excel(173, "Criminology")


In [15]:
set_row_excel(181, "Chemistry")

In [16]:
set_row_excel(202, "Midwifery")

In [17]:
set_row_excel(211, "Social Work")
set_row_excel(213, "Pharmacy")

In [18]:
set_row_excel(244, "Electrical Engineering")
set_row_excel(251, "Accountancy")
set_row_excel(253, "Accountancy")
set_row_excel(262, "Criminology")

In [19]:
set_row_excel(274, "Customs Brokers")

In [20]:
set_row_excel(292, "Dentistry")
set_row_excel(294, "Mechanical Engineering")
set_row_excel(295, "Professional Teachers")
set_row_excel(314, "Architecture")

In [21]:
set_row_excel(319, "Professional Teachers")
set_row_excel(334, "Electrical Engineering")
set_row_excel(335, "Electrical Engineering")
set_row_excel(338, "Electronics Engineering")


In [22]:
set_row_excel(348, "Electronics Engineering")
set_row_excel(357, "Medical Technology")
set_row_excel(359, "Professional Teachers")

In [23]:
set_row_excel(375, "Civil Engineering")
set_row_excel(386, "Accountancy")
set_row_excel(392, "Professional Teachers")
set_row_excel(397, "Agricultural Engineering")


In [24]:
set_row_excel(406, "Agricultural Engineering")
set_row_excel(408, "Civil Engineering")
set_row_excel(417, "Professional Teachers")
set_row_excel(422, "Food Technology")
set_row_excel(423, "Food Technology")

In [25]:
set_row_excel(436, "Food Technology")
set_row_excel(439, "Food Technology")
set_row_excel(441, "Professional Teachers")
set_row_excel(443, "Accountancy")
set_row_excel(446, "Forestry")


In [26]:
set_row_excel(465, "Professional Teachers")
set_row_excel(466, "Professional Teachers")
set_row_excel(469, "Physical Therapy")

In [27]:
set_row_excel(501, "Professional Teachers")
set_row_excel(504, "Midwifery")
set_row_excel(510, "Pharmacy")
set_row_excel(513, "Civil Engineering")


In [ ]:
set_row_excel(534, "Chemistry")
set_row_excel(540, "Fisheries Technology")
set_row_excel(542, "Professional Teachers")
set_row_excel(544, "Mechanical Engineering")
set_row_excel(550, "Architecture")
set_row_excel(551, "Architecture")
set_row_excel(559, "Accountancy")

In [29]:
set_row_excel(572, "Accountancy")  
set_row_excel(574, "Architecture")      
set_row_excel(578, "Accountancy")     
set_row_excel(587, "Accountancy")  
set_row_excel(589, "Geology")             
set_row_excel(593, "Landscape Architecture")  
set_row_excel(595, "Electrical Engineering")    

In [30]:
set_row_excel(606, "Professional Teachers")
set_row_excel(608, "Library Science")
set_row_excel(614, "Architecture")

In [31]:
set_row_excel(627, "Criminology")
set_row_excel(633, "Library Science")
set_row_excel(636, "Library Science")
set_row_excel(637, "Aeronautical Engineering")
set_row_excel(638, "Agricultural Engineering")
set_row_excel(639, "Agricultural Engineering")
set_row_excel(641, "Architecture")
set_row_excel(642, "Chemistry")
set_row_excel(643, "Criminology")
set_row_excel(644, "Dentistry")
set_row_excel(645, "Nutrition and Dietetics")
set_row_excel(646, "Occupational Therapy")
set_row_excel(647, "Optometry")
set_row_excel(648, "Pharmacy")
set_row_excel(649, "Physical Therapy")
set_row_excel(650, "Psychology")
set_row_excel(651, "Respiratory Therapy")

In [32]:
set_row_excel(660, "Professional Teachers")
set_row_excel(661, "Agriculture")
set_row_excel(674, "Mechanical Engineering")

In [33]:
set_row_excel(686, "Accountancy")
set_row_excel(690, "Mechanical Engineering")

In [34]:
set_row_excel(708, "Metallurgical Engineering")
set_row_excel(712, "Professional Teachers")
set_row_excel(717, "Mechanical Engineering")
set_row_excel(718, "Professional Teachers")

In [35]:
set_row_excel(728, "Medical Technology")
set_row_excel(742, "Mining Engineering")

In [36]:
set_row_excel(765, "Professional Teachers")
set_row_excel(774, "Nursing")
set_row_excel(775, "Professional Teachers")
set_row_excel(777, "Nutrition and Dietetics")
set_row_excel(778, "Food Technology")
set_row_excel(785, "Nursing")

In [37]:
set_row_excel(792, "Physical Therapy")
set_row_excel(807, "Medical Technology")
set_row_excel(811, "Accountancy")

In [38]:
set_row_excel(819, "Pharmacy")

In [39]:
set_row_excel(855, "Professional Teachers")
set_row_excel(859, "Chemistry")
set_row_excel(860, "Mechanical Engineering")
set_row_excel(865, "Accountancy")
set_row_excel(866, "Civil Engineering")
set_row_excel(868, "Food Technology")

In [40]:
set_row_excel(885, "Psychology")
set_row_excel(902, "Chemistry")
set_row_excel(903, "Civil Engineering")
set_row_excel(908, "Food Technology")
set_row_excel(909, "Radiologic Technology")

In [41]:
set_row_excel(919, "Forestry")
set_row_excel(920, "Naval Architecture and Marine Engineering")
set_row_excel(921, "Nursing")
set_row_excel(922, "Radiologic Technology")
set_row_excel(923, "Professional Teachers")
set_row_excel(926, "Professional Teachers")
set_row_excel(927, "Agricultural Engineering")
set_row_excel(937, "Electronics Engineering")

In [42]:
set_row_excel(963, "Medicine")
set_row_excel(969, "Professional Teachers")

In [43]:
set_row_excel(993, "Civil Engineering")
set_row_excel(1003, "Social Work")
set_row_excel(1004, "Social Work")
set_row_excel(1014, "Professional Teachers")
set_row_excel(1015, "Professional Teachers")
set_row_excel(1016, "Professional Teachers")
set_row_excel(1017, "Medicine")
set_row_excel(1019, "Professional Teachers")


In [44]:
set_row_excel(1024, "Mechanical Engineering")
set_row_excel(1035, "Architecture")
set_row_excel(1043, "Civil Engineering")
set_row_excel(1048, "Accountancy")


In [45]:
set_row_excel(1056, "Accountancy")
set_row_excel(1077, "Professional Teachers")
set_row_excel(1078, "Professional Teachers")
set_row_excel(1079, "Professional Teachers")


In [46]:
set_row_excel(1099, "Professional Teachers")
set_row_excel(1104, "Forestry")
set_row_excel(1107, "Professional Teachers")
set_row_excel(1120, "Occupational Therapy")


In [47]:
set_row_excel(1133, "Sanitary Engineering")
set_row_excel(1135, "Sanitary Engineering")

Saving the file

In [48]:
save_updated("MCA_PRC_TAGGING.csv")


'MCA_PRC_TAGGING.csv'